# Welcome to the Day 2 Lab!


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">Just before we get started --</h2>
            <span style="color:#f71;">I thought I'd take a second to point you at this page of useful resources for the course. This includes links to all the slides.<br/>
            <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">https://edwarddonner.com/2024/11/13/llm-engineering-resources/</a><br/>
            Please keep this bookmarked, and I'll continue to add more useful links there over time.
            </span>
        </td>
    </tr>
</table>

## First - let's talk about the Chat Completions API

1. The simplest way to call an LLM
2. It's called Chat Completions because it's saying: "here is a conversation, please predict what should come next"
3. The Chat Completions API was invented by OpenAI, but it's so popular that everybody uses it!

### We will start by calling OpenAI again - but don't worry non-OpenAI people, your time is coming!


In [1]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-"):
    print("An API key was found, but it doesn't start sk-; please check you're using the right key - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")


API key found and looks good so far!


## Do you know what an Endpoint is?

If not, please review the Technical Foundations guide in the guides folder

And, here is an endpoint that might interest you...

In [2]:
import requests

headers = {"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"}

payload = {
    "model": "gpt-5-nano",
    "messages": [
        {"role": "user", "content": "Tell me a fun fact about horses"}]
}

payload

{'model': 'gpt-5-nano',
 'messages': [{'role': 'user', 'content': 'Tell me a fun fact about horses'}]}

In [3]:
response = requests.post(
#    "https://api.openai.com/v1/chat/completions",
    "https://openrouter.ai/api/v1/chat/completions",
    headers=headers,
    json=payload
)

response.json()

{'id': 'gen-1762697011-rBqw5FVbZyINK2ffSfZo',
 'provider': 'OpenAI',
 'model': 'openai/gpt-5-nano',
 'object': 'chat.completion',
 'created': 1762697011,
 'choices': [{'logprobs': None,
   'finish_reason': 'stop',
   'native_finish_reason': 'completed',
   'index': 0,
   'message': {'role': 'assistant',
    'content': 'Fun fact: Horses can sleep both standing up and lying down. They have a special “stay apparatus” in their legs that lets them lock their joints and doze while standing, though they’ll still lie down for deeper REM sleep when they feel safe.',
    'refusal': None,
    'reasoning': '**Sharing fun facts about horses**\n\nI need to give a fun fact about horses since the user asked for one. I could mention a few interesting things, like how horses have strong long-term memory and can recall people and places for years. There\'s also the idea that they might sense earthquakes, which some studies suggest. A great fun fact to share is that horses can sleep both standing up and l

In [4]:
response.json()["choices"][0]["message"]["content"]

'Fun fact: Horses can sleep both standing up and lying down. They have a special “stay apparatus” in their legs that lets them lock their joints and doze while standing, though they’ll still lie down for deeper REM sleep when they feel safe.'

# What is the openai package?

It's known as a Python Client Library.

It's nothing more than a wrapper around making this exact call to the http endpoint.

It just allows you to work with nice Python code instead of messing around with janky json objects.

But that's it. It's open-source and lightweight. Some people think it contains OpenAI model code - it doesn't!


In [5]:
# Create OpenAI client

from openai import OpenAI
openai = OpenAI(
    base_url="https://openrouter.ai/api/v1",
)

response = openai.chat.completions.create(model="gpt-5-nano", messages=[{"role": "user", "content": "Tell me a fun fact"}])

response.choices[0].message.content



'Fun fact: An octopus has three hearts—two pump blood through the gills, and a third pumps it to the rest of the body. When it swims, the third heart slows down, which is why they mostly crawl rather than swim long distances. Want another fun fact?'

## And then this great thing happened:

OpenAI's Chat Completions API was so popular, that the other model providers created endpoints that are identical.

They are known as the "OpenAI Compatible Endpoints".

For example, google made one here: https://generativelanguage.googleapis.com/v1beta/openai/

And OpenAI decided to be kind: they said, hey, you can just use the same client library that we made for GPT. We'll allow you to specify a different endpoint URL and a different key, to use another provider.

So you can use:

```python
gemini = OpenAI(base_url="https://generativelanguage.googleapis.com/v1beta/openai/", api_key="AIz....")
gemini.chat.completions.create(...)
```

And to be clear - even though OpenAI is in the code, we're only using this lightweight python client library to call the endpoint - there's no OpenAI model involved here.

If you're confused, please review Guide 9 in the Guides folder!

And now let's try it!

In [6]:
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"

google_api_key = os.getenv("GOOGLE_API_KEY")

if not google_api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not google_api_key.startswith("AIz"):
    print("An API key was found, but it doesn't start AIz")
else:
    print("API key found and looks good so far!")



API key found and looks good so far!


In [7]:
gemini = OpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)

response = gemini.chat.completions.create(model="gemini-2.5-pro", messages=[{"role": "user", "content": "Tell me a fun fact"}])

response.choices[0].message.content

'A group of flamingos is called a **flamboyance**'

## And Ollama also gives an OpenAI compatible endpoint

...and it's on your local machine!

If the next cell doesn't print "Ollama is running" then please open a terminal and run `ollama serve`

In [8]:
requests.get("http://localhost:11434").content

b'Ollama is running'

### Download llama3.2 from meta

Change this to llama3.2:1b if your computer is smaller.

Don't use llama3.3 or llama4! They are too big for your computer..

In [9]:
!ollama pull llama3.2

pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest 
pulling dde5aa3fc5ff: 100% ▕██████████████████▏ 2.0 GB                         
pulling 966de95ca8a6: 100% ▕██████████████████▏ 1.4 KB                         
pulling fcc5a6bec9da: 100% ▕██████████████████▏ 7.7 KB                         
pulling a70ff7e570d9: 100% ▕██████████████████▏ 6.0 KB                         
pulling 56bb8bd477a5: 100% ▕██████████████████▏   96 B                         
pulling 34bb5ab01051: 100% ▕██████████████████▏  561 B                         
verifying sha256 digest 
writing manifest 
success 


In [10]:
OLLAMA_BASE_URL = "http://localhost:11434/v1"

ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')


In [11]:
# Get a fun fact

response = ollama.chat.completions.create(model="llama3.2", messages=[{"role": "user", "content": "Tell me a fun fact"}])

response.choices[0].message.content

"Here's one:\n\nDid you know that honey never spoils? Archaeologists have found pots of honey in ancient Egyptian tombs that are over 3,000 years old and still perfectly edible. Honey's longevity is due to its unique composition, which means it has antimicrobial properties that prevent the growth of bacteria, yeast, and mold. This makes it a virtually immortal food!"

In [12]:
# Now let's try deepseek-r1:1.5b - this is DeepSeek "distilled" into Qwen from Alibaba Cloud

!ollama pull deepseek-r1:1.5b

pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest 
pulling aabd4debf0c8: 100% ▕██████████████████▏ 1.1 GB                         
pulling c5ad996bda6e: 100% ▕██████████████████▏  556 B                         
pulling 6e4c38e1172f: 100% ▕██████████████████▏ 1.1 KB                         
pulling f4d24e9138dd: 100% ▕██████████████████▏  148 B                         
pulling a85fe2a2e58e: 100% ▕██████████████████▏  487 B                         
verifying sha256 digest 
writing manifest 
success 


In [13]:
response = ollama.chat.completions.create(model="deepseek-r1:1.5b", messages=[{"role": "user", "content": "Tell me a fun fact"}])

response.choices[0].message.content

"Sure! Here's a fun fact about something I’ve always admired about the ocean: The Great barrier Reef in Australia is considered the “biggest man-made natural barrier against rising water levels” thanks to its alkaline environment. This unique ecosystem is home to a variety of marine life that helps shape the Reef into such an impressive feature.\n\nAdditionally, you might enjoy hearing about something as ancient as the Alhambra, a stunning stone-built labyrinth in Spain called The Crown. It’s not just for men; it's for everyone with its intricate design and grandeur, much like how we should appreciate our planet on a larger scale!"

# HOMEWORK EXERCISE ASSIGNMENT

Upgrade the day 1 project to summarize a webpage to use an Open Source model running locally via Ollama rather than OpenAI

You'll be able to use this technique for all subsequent projects if you'd prefer not to use paid APIs.

**Benefits:**
1. No API charges - open-source
2. Data doesn't leave your box

**Disadvantages:**
1. Significantly less power than Frontier Model

## Recap on installation of Ollama

Simply visit [ollama.com](https://ollama.com) and install!

Once complete, the ollama server should already be running locally.  
If you visit:  
[http://localhost:11434/](http://localhost:11434/)

You should see the message `Ollama is running`.  

If not, bring up a new Terminal (Mac) or Powershell (Windows) and enter `ollama serve`  
And in another Terminal (Mac) or Powershell (Windows), enter `ollama pull llama3.2`  
Then try [http://localhost:11434/](http://localhost:11434/) again.

If Ollama is slow on your machine, try using `llama3.2:1b` as an alternative. Run `ollama pull llama3.2:1b` from a Terminal or Powershell, and change the code from `MODEL = "llama3.2"` to `MODEL = "llama3.2:1b"`

In [14]:
from scraper import fetch_website_contents
from IPython.display import Markdown

def beautify_output(response):
    display(Markdown(response))


# Define our system prompt - you can experiment with this later, changing the last sentence to 'Respond in markdown in Spanish."
system_prompt = """
You are a creative writer that optimizes the content of a website to be more engaging.
You are given the contents of ann existing website and are to rewrite it without altering the original meaning.

Make sure to focus on being engaging and interesting, and writing the information in a way that is easy to understand.
You ignore navigation text, and focus on the content. Do not wrap the markdown in a code block - respond just with the markdown for the new content blocks.
"""

# Define our user prompt
user_prompt_prefix = """
Here are the contents of a website.

Please rewrite the content to be more engaging and interesting, and writing the information in a way that is easy to understand.

"""

In [15]:

website_content = fetch_website_contents("https://edwarddonner.com")

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt_prefix + website_content}
]

OLLAMA_BASE_URL = "http://localhost:11434/v1"
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')

response = ollama.chat.completions.create(model="llama3.2", messages=messages)
response.choices[0].message.content

"# Welcome to My World: Exploring the Intersection of AI and Human Potential\n\n## About Me\n\nHey there, fellow code enthusiasts and AI aficionados! I'm Ed, a passionate writer, coder, and DJ who loves experimenting with large language models (LLMs). When I'm not geeking out over AI or music production, you can find me nodding my head to insightful articles on Hacker News. As the co-founder and CTO of [Nebula.io](https://www.nebula-io.com/), I'm dedicated to harnessing the power of AI to unlock human potential.\n\nIn a nutshell, our mission is to empower people to discover their purpose and passions using cutting-edge AI solutions. Recruiters already trust our technology to source, understand, engage with, and manage top talent. Our journey began when I founded [Untapt](https://untapt.com/) in 2021, which was later acquired by Nebula.io.\n\n## What We Achieve\n\nWe've developed groundbreaking LLMs specifically tailored for the talents industry and patented our innovative matching mode

In [16]:
beautify_output(response.choices[0].message.content)

# Welcome to My World: Exploring the Intersection of AI and Human Potential

## About Me

Hey there, fellow code enthusiasts and AI aficionados! I'm Ed, a passionate writer, coder, and DJ who loves experimenting with large language models (LLMs). When I'm not geeking out over AI or music production, you can find me nodding my head to insightful articles on Hacker News. As the co-founder and CTO of [Nebula.io](https://www.nebula-io.com/), I'm dedicated to harnessing the power of AI to unlock human potential.

In a nutshell, our mission is to empower people to discover their purpose and passions using cutting-edge AI solutions. Recruiters already trust our technology to source, understand, engage with, and manage top talent. Our journey began when I founded [Untapt](https://untapt.com/) in 2021, which was later acquired by Nebula.io.

## What We Achieve

We've developed groundbreaking LLMs specifically tailored for the talents industry and patented our innovative matching model. Our award-winning platform has garnered widespread recognition and happy customers.

Stay connected with me on my latest projects and adventures!

* Read my recent AI insights: [September 15, 2025: AI in Production] or check out these topics: [May 28, 2025: Be an AI Engineer], [May 18, 2025: The Curriculum] and more.
* Join the conversation on social media: LinkedIn | Twitter | Facebook
* Sign up for our newsletter to stay updated on news & updates

---

Let me know if you would like me to do anything else!